In [1]:
global K
K = 2
global seed
seed = 123

import pandas as pd
import yfinance as yf
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.optimize import minimize



from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

In [ ]:
# Setup IBM account to later connect to the cloud to then connect to their QPUs.
def setup_ibm_acc():
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token="HxGhEHlaU6NNVZJ_sx6EJRsWT4K62QIpvNzkOytkz2N2",
        overwrite=True
    )

setup_ibm_acc()

num_layers=3

In [ ]:
# Connect to IBM computer or 'quantum platform' & use the least busy one.
def connect_QPU():
    service = QiskitRuntimeService(channel='ibm_quantum_platform')

    backend = service.least_busy(simulator=False, operational=True, min_num_qubits=8)
    status = backend.status()
    print(status)

    return service, backend

acc = connect_QPU()

management.get:WARNING:2025-10-11 08:41:45,472: Loading default saved account
qiskit_runtime_service.__init__:WARNING:2025-10-11 08:41:50,958: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Quantathon. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2025-10-11 08:41:51,369: Loading instance: Quantathon, plan: open
qiskit_runtime_service.backends:WARNING:2025-10-11 08:41:52,424: Using instance: Quantathon, plan: open


In [ ]:
# Data pre-processing
tickers_path = 'prices.csv'

if os.path.exists(tickers_path):
    ticker_data = pd.read_csv(tickers_path)
else:
    tickers = ['GOOG', 'XOM', 'AAPL', 'AMZN', 'GLD', 'DUK', 'SO', 'AEP']
    date_range = ['2017-01-01', '2017-03-01']
    ticker_data = yf.download(
        tickers, start=date_range[0], end=date_range[1],
        group_by="ticker", auto_adjust=True
    )
    ticker_data = ticker_data.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()
    ticker_data.to_csv(tickers_path, index=False)

price_df = ticker_data.pivot(index="Date", columns="Ticker", values="Close")
log_returns = np.log(price_df / price_df.shift(1)).dropna()
data = log_returns


In [5]:
def solve_covariance_matrix(data):
    """Calculate covariance matrix for given sample data"""
    J = np.cov(data, rowvar=False)
    return J


In [6]:
def annealing_qiskit_ibm(gamma_list, beta_list, J, N, num_layers, backend, return_best=False):
    qc = QuantumCircuit(N)
    
    # Hadamard gates
    for i in range(N):
        qc.h(i)
    qc.barrier()
    
    # Apply QAOA layers
    for i in range(num_layers):
        gamma = gamma_list[i]
        beta = beta_list[i]
        apply_cost_mixer_layers(N, J, qc, gamma, beta)
    
    # Measure the circuit
    qc.measure_all()
    
    # Transpile circuit for target hardware
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    transpiled_qc = pm.run(qc)
    
    # Run on IBM hardware using SamplerV2
    sampler = Sampler(backend)
    
    # Run with appropriate shots
    shots = 1000  # Adjust based on your needs
    
    job = sampler.run([transpiled_qc], shots=shots)
    
    # Wait for result
    result = job.result()
    
    # Extract counts from SamplerV2 result
    pub_result = result[0]
    counts_dict = pub_result.data.meas.get_counts()
    
    # Calculate expectation and find best solution
    total_shots = sum(counts_dict.values())
    expected = 0
    best_objective = -np.inf
    best_bitstring = None
    
    for bitstring, count in counts_dict.items():
        b = np.array([1 if bit == '0' else -1 for bit in bitstring[::-1]])
        energy = b.T @ J @ b
        expected += energy * (count / total_shots)
        
        if energy > best_objective:
            best_objective = energy
            best_bitstring = bitstring
    
    if return_best:
        return expected, best_bitstring, best_objective
    return expected



In [7]:
def apply_cost_mixer_layers(N, J, qc, gamma, beta):
    """
    Applies main layers of circuit (QAOA layers)
    """
    # Problem Hamiltonian: ZZ couplings
    for i in range(N):
        for j in range(i+1, N):
            if J[i,j] != 0:
                qc.cx(i,j)
                qc.rz(2*gamma*(J[i,j]), j)
                qc.cx(i,j)
    
    # Diagonal terms
    for i in range(N):
        if J[i,i] != 0:
            qc.rz(2*gamma*(J[i,i]), i)
    
    qc.barrier()
    
    # Mixer Hamiltonian
    for i in range(N):
        qc.rx(2*beta, i)
    
    qc.barrier()

In [ ]:
def obj_func_ibm(params, J, N, backend):
    """
    Modify objective function to pass the backend into the main function as a way to use the backend to run the code.
    """
    gamma_list = params[:3]
    beta_list = params[3:]
    
    expected = annealing_qiskit_ibm(gamma_list, beta_list, J, N, num_layers, backend)
    print(expected)
    return -expected  # Negative because we're maximizing

In [9]:
def run_annealing_qiskit_ibm(data, backend):    
    # Initial parameters
    gamma_list = [0.5, 0.5, 0.5]
    beta_list = [0.3, 0.3, 0.3]
    N = 8  # Number of qubits
    
    # Compute covariance
    J = solve_covariance_matrix(data)
    J = (J + J.T) / 2
    
    # Scale for numerical stability
    scale_factor = 1000
    J_scaled = J * scale_factor
    
    params = np.concatenate([gamma_list, beta_list])
    
    # Run optimization
    optimization = minimize(
        obj_func_ibm,
        params,
        args=(J_scaled, N, backend),
        method='COBYLA',
        options={'maxiter': 50}
    )
    
    best_gamma = optimization.x[:3]
    best_beta = optimization.x[3:]
    
    # Final run to get complete results
    print("\nRunning final evaluation...")
    expectation, best_bitstring, best_objective = annealing_qiskit_ibm(
        best_gamma, best_beta, J_scaled, N, num_layers, backend, return_best=True
    )
    
    # Unscale
    expectation_unscaled = expectation / scale_factor
    best_objective_unscaled = best_objective / scale_factor
    
    # Get best b vector
    best_b = np.array([1 if bit == '0' else -1 for bit in best_bitstring[::-1]])
    
    print("\n=== QAOA RESULTS ===")
    print(f"Optimal gamma: {best_gamma}")
    print(f"Optimal beta: {best_beta}")
    print(f"Expectation value: {expectation_unscaled:.6f}")
    print(f"Best solution objective: {best_objective_unscaled:.6f}")
    print(f"Best bitstring: {best_bitstring}")
    print(f"Best portfolio vector b: {best_b}")
    
    return best_gamma, best_beta, expectation_unscaled, best_bitstring, best_b

run_annealing_qiskit_ibm(data, acc[1])
    

0.6356097179401985
0.6619659537680909
0.728895016033652
0.6887710482415383
0.646449633041538
0.6309420539268741
0.6067960590679111
0.6238649318297188
0.6198066233709478
0.6822615774674129
0.6296722596683224
0.678376899260398
0.6779828780016134
0.6063902296900444
0.6323903887316191
0.7361054207807105
0.7416541358414381
0.6782683811591436
0.7109361499960748
0.7073289422982316
0.7020993626973637
0.6763416887122725
0.7063650295665402
0.6837867249616044
0.694751627517454
0.6898363233261662
0.720518153395812
0.6807870151054144
0.6485167594290684
0.7222636119835886
0.6904375753231812
0.7095874544085654
0.671465526599336
0.691587131256452
0.698028466432628
0.7191117769830759
0.6709289694172034
0.7185321041567273
0.6972582215744448
0.6778181017442475
0.634356599654932
0.7057704004683526
0.714222467041407
0.7205908088060876
0.7156112470948333
0.6945889439601368
0.6787527434853277
0.7154389069124314
0.6958175395233424
0.7118470755169942

Running final evaluation...

=== QAOA RESULTS ===
Optimal g

(array([1.48477027, 1.52638314, 0.44399154]),
 array([0.31609915, 0.2185933 , 0.20737142]),
 np.float64(0.0007014235489303062),
 '00100101',
 array([-1,  1, -1,  1,  1, -1,  1,  1]))